In [ ]:
import os, glob, re
from pathlib import Path
from joblib import Parallel, delayed

import numpy as np
import pandas as pd
from scipy import stats, sparse
from scipy.stats import ttest_rel, ttest_ind
from scipy.sparse.csgraph import connected_components
from statsmodels.stats.multitest import multipletests

from nilearn import plotting, image

import matplotlib.pyplot as plt
from bct import nbs as bct_nbs

In [ ]:
HPC        = '/nfs/roberts/pi/pi_il77/Nachshon/'
ATLAS      = 'difumo'
DATA_DIR   = f'/nfs/roberts/pi/pi_il77/Nachshon/PSUB/preProc/corMatrix_byTrial_pr/'
ATLAS_IMG  = f'{HPC}ROI/Atlas/difumo_2mm_maps_in_funcgrid.nii.gz' 
ATLAS_DIC  = f'{HPC}ROI/Atlas/labels_256_dictionary.csv'
OUT_DIR    = "./nbsFinal"
CONDITIONS = ['FULL', 'HOT60']
GROUPS     = ['PTSD', 'HC']
META_CSV   = '../../BehavioralData/clinical.csv'  
EPS        = 1e-7
K          = 50
N_JOBS     = 4 
Threshold  = 3.5


os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Define file path
coords_file = os.path.join(OUT_DIR, f"{ATLAS}_node_coords.csv")

if os.path.exists(coords_file):
    print(f"Loading coordinates from {coords_file}...")
    # Load the saved coordinates
    node_coords = np.loadtxt(coords_file, delimiter=',')
else:
    print(f"Calculating coordinates for {ATLAS} (this may take a while)...")
    labels_img = image.load_img(ATLAS_IMG)
    
    if ATLAS.lower() == 'difumo':
        # 4D probabilistic logic
        node_coords = plotting.find_probabilistic_atlas_cut_coords(labels_img)
    else:
        # 3D deterministic logic
        node_coords = plotting.find_parcellation_cut_coords(labels_img)
    
    # Save to OUT_DIR
    np.savetxt(coords_file, node_coords, delimiter=',')
    print(f"Coordinates saved to {coords_file}")

p = len(node_coords)
print(f"ROIs (from atlas): {p}")

In [ ]:
all_files = sorted(glob.glob(os.path.join(DATA_DIR, f"*_{ATLAS}.csv")))

by_sub = {}
for f in all_files:
    base = os.path.basename(f).replace(f"_{ATLAS}.csv","")
    sub, _, cond = base.split("_", 2)
    sub = sub.split('-')[1]
    if cond in CONDITIONS:
        by_sub.setdefault(sub, {})[cond] = f


subs = sorted([s for s,d in by_sub.items() if all(c in d for c in CONDITIONS)])
print(f"Subjects with both conditions: n={len(subs)}")
assert len(subs) >= 2, "Need at least 2 subjects."

In [ ]:
meta = pd.read_csv(META_CSV)
meta["sub_id"] = meta["sub_id"].astype(str).str.strip()
meta["group"]  = meta["group"].str.upper().str.strip()

# subjects present in matrices
subs_all = sorted([s for s, d in by_sub.items() if any(c in d for c in CONDITIONS)])


# ensure normalized meta column is string
meta_ids = set(meta["sub_id"].astype(str))

missing_in_meta = [s for s in subs_all if s not in meta_ids]
if missing_in_meta:
    print("Subjects present in matrices but missing behavior:", missing_in_meta)


In [ ]:
# subjects in meta but missing any matrices (per condition we’ll check below)
present_in_meta = meta[meta["sub_id"].isin(by_sub.keys())].copy()
groups = present_in_meta.set_index("sub_id")["group"].to_dict()  # {sub: PTSD/HC}

In [ ]:
def load_mat(path):
    df  = pd.read_csv(path)               # first column is 'roi' (0..49)
    arr = df.drop(columns=df.columns[0]).to_numpy()
    arr = np.clip(arr, -1+EPS, 1-EPS)
    return np.arctanh(arr)                # Fisher-z

def stack_group(condition, group_name):
    subs = [s for s,g in groups.items() if g == group_name and condition in by_sub.get(s, {})]
    Z = np.stack([load_mat(by_sub[s][condition]) for s in subs], axis=0) if subs else None
    return subs, Z

def get_group_deltas(group_name, cond1, cond2):
    """
    Returns (subjects, Delta_Z) where Delta_Z = Z(cond1) - Z(cond2)
    Only includes subjects who possess BOTH condition files.
    """
    # 1. Identify subjects in this group who have BOTH conditions
    valid_subs = [
        s for s, g in groups.items()
        if g == group_name 
        and cond1 in by_sub.get(s, {}) 
        and cond2 in by_sub.get(s, {})
    ]
    valid_subs.sort()
    
    if not valid_subs:
        print(f"Warning: No valid subjects found for {group_name} with both conditions.")
        return [], None

    print(f"Processing Delta for {group_name}: n={len(valid_subs)} ({cond1} minus {cond2})")

    # 2. Load matrices (load_mat applies Fisher-Z transform)
    Z1 = np.stack([load_mat(by_sub[s][cond1]) for s in valid_subs], axis=0)
    Z2 = np.stack([load_mat(by_sub[s][cond2]) for s in valid_subs], axis=0)

    # 3. Compute Difference
    Z_delta = Z1 - Z2
    
    return valid_subs, Z_delta

In [ ]:
subs_ptsd, D_ptsd = get_group_deltas('PTSD', CONDITIONS[0], CONDITIONS[1])
subs_hc,   D_hc   = get_group_deltas('HC',   CONDITIONS[0], CONDITIONS[1])

In [ ]:
def run_nbs_interaction(Z_A, Z_B, label_A, label_B, condition_label, node_coords, 
                        thresh=3.1, k=5000, fig_dpi=300, atlas_dict_csv=None):
    """
    Directly accepts Delta arrays Z_A and Z_B.
    """
    try:
        # 1. Setup Data
        n1, n2 = len(Z_A), len(Z_B)
        p = Z_A.shape[1]

        # 2. Run NBS (Unpaired T-test on Deltas)
        A_bct = np.moveaxis(Z_A, 0, -1)
        B_bct = np.moveaxis(Z_B, 0, -1)
        
        pval, adj, _ = bct_nbs.nbs_bct(A_bct, B_bct, thresh=thresh, k=k, 
                                       paired=False, verbose=False, tail='both')

        # 3. Process Adjacency
        if isinstance(adj, list):
            adj_mask = np.zeros((p, p), dtype=int)
            for a in adj: adj_mask |= (np.array(a, dtype=int) != 0)
        else:
            adj_mask = (np.array(adj, dtype=int) != 0)

        n_edges = int(adj_mask.sum() // 2)
        comp_pvals = np.atleast_1d(pval)

        if n_edges == 0:
            return {"condition": condition_label, "p_val": np.min(comp_pvals), "n_edges": 0, "avg_d": 0, "comparison": f"{label_A} vs {label_B}"}

        # 4. Interaction (Delta-Delta Contrast)
        mean_A = Z_A.mean(axis=0)
        mean_B = Z_B.mean(axis=0)
        cont_mat = mean_A - mean_B
        
        iu = np.triu_indices(p, k=1)
        mask_flat = adj_mask[iu]
        
        vals_A = Z_A[:, iu[0][mask_flat], iu[1][mask_flat]]
        vals_B = Z_B[:, iu[0][mask_flat], iu[1][mask_flat]]
        
        tvals, pvals_t = stats.ttest_ind(vals_A, vals_B, axis=0, equal_var=False)
        
        # Pooled SD for Cohen's d
        var_A, var_B = np.var(vals_A, axis=0, ddof=1), np.var(vals_B, axis=0, ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * var_A + (n2 - 1) * var_B) / (n1 + n2 - 2))
        d_per_edge = (np.mean(vals_A, axis=0) - np.mean(vals_B, axis=0)) / pooled_sd
        avg_d = np.mean(np.abs(d_per_edge))

        # 5. Save Edge Table
        df_sig = pd.DataFrame({
            "roi_i": iu[0][mask_flat] + 1,
            "roi_j": iu[1][mask_flat] + 1,
            "mean_Delta_A": mean_A[iu][mask_flat],
            "mean_Delta_B": mean_B[iu][mask_flat],
            "Interaction_Diff": cont_mat[iu][mask_flat],
            "t_stat": tvals,
            "cohen_d": d_per_edge,
            "p_unc": pvals_t
        })

        if atlas_dict_csv:
            atlas = pd.read_csv(atlas_dict_csv).rename(columns={"Component": "Comp", "Difumo_names": "Label"})
            df_sig = df_sig.merge(atlas.add_suffix("_i"), left_on="roi_i", right_on="Comp_i", how="left")
            df_sig = df_sig.merge(atlas.add_suffix("_j"), left_on="roi_j", right_on="Comp_j", how="left")
            

        tag = f"INTERACTION_{condition_label}_{label_A}_vs_{label_B}"
        df_sig.to_csv(f"{OUT_DIR}/{tag}_edges.csv", index=False)

        # 6. Plotting
        adj_signed = cont_mat * adj_mask
        vmax = np.nanmax(np.abs(adj_signed)) or 0.1
        
        fig = plt.figure(figsize=(8, 6))
        plotting.plot_connectome(adj_signed, node_coords, edge_threshold=0.0,
                                 edge_cmap=plt.cm.RdBu_r, edge_vmin=-vmax, edge_vmax=vmax,
                                 colorbar=True, title=f"Interaction: {tag}\nAvg d = {avg_d:.3f}", figure=fig)
        plt.savefig(f"{OUT_DIR}/{tag}_connectome.png", dpi=fig_dpi)
        plt.close(fig)

        return {
            "condition": condition_label,
            "comparison": f"{label_A} vs {label_B}",
            "p_val": np.min(comp_pvals),
            "n_edges": n_edges,
            "avg_d": avg_d,
            "adj_mask_union": adj_mask,
            "contrast_r": cont_mat,
            "labels": {"A": (condition_label, label_A), "B": (condition_label, label_B)}
        }
    except Exception as e:
        return {"condition": condition_label, "error": str(e)}

In [ ]:
def run_nbs(
    node_coords,
    condition1,
    group1,
    condition2="",
    group2="",
    out_dir=OUT_DIR,
    thresh=2.5,        # primary t threshold on edgewise t
    tail="both",       # 'both' | 'left' | 'right'
    k=5000,            # permutations
    paired=False,      # within-subjects => True; between-groups => False
    fig_dpi=300,
    verbose=False,
    atlas_dict_csv=ATLAS_DIC,    
    atlas_component_col="Component",
    atlas_name_col="Difumo_names"
):
    """
    Run NBS (bctpy), compute edgewise stats, attach DiFuMo labels+coords, and save CSVs.

    Requires:
      - stack_group(condition, group) -> (subs, Z) with Z: (n, p, p) Fisher-z per subject
      - node_coords: array-like (p, 3)
      - atlas_dict_csv (optional): CSV with columns [Component, Difumo_names]
    """
    os.makedirs(out_dir, exist_ok=True)

    # --- design wiring ---
    if paired:
        if not group2:
            group2 = group1
        if not condition2:
            raise ValueError("For paired=True, provide condition2.")
        if group2 != group1:
            raise ValueError("For paired=True, group2 must equal group1.")
    else:
        if not condition2:
            condition2 = condition1
        if not group2:
            raise ValueError("For paired=False, provide group2.")
        if condition2 != condition1:
            raise ValueError("For paired=False, condition2 must equal condition1.")

    # --- load ---
    subs_1, Z_1 = stack_group(condition1, group1)
    subs_2, Z_2 = stack_group(condition2, group2)
    assert (Z_1 is not None) and (Z_2 is not None), "No data for one or both cells."

    p = Z_1.shape[1]
    assert Z_2.shape[1] == p == len(node_coords), "ROI mismatch with node_coords."

    # align for paired
    if paired:
        common = sorted(set(subs_1) & set(subs_2))
        if len(common) < 2:
            raise ValueError(f"Paired design needs ≥2 overlapping subjects; got {len(common)}.")
        m1 = {s:i for i,s in enumerate(subs_1)}
        m2 = {s:i for i,s in enumerate(subs_2)}
        Z_1 = np.stack([Z_1[m1[s]] for s in common], axis=0)
        Z_2 = np.stack([Z_2[m2[s]] for s in common], axis=0)
        subs_1 = subs_2 = common

    if verbose:
        print(f"[NBS] {condition1} {group1} n={len(subs_1)}, {condition2} {group2} n={len(subs_2)}, p={p}")
        print(f"[NBS] thresh={thresh}, tail={tail}, perms={k}, paired={paired}")

    # --- NBS (bctpy wants [p,p,subs]) ---
    A = np.moveaxis(Z_1, 0, -1)
    B = np.moveaxis(Z_2, 0, -1)
    pvals_nbs, adj, _ = bct_nbs.nbs_bct(A, B, thresh=thresh, tail=tail, k=k, paired=paired, verbose=verbose)

    if isinstance(adj, list):
        adj_mask = np.zeros((p, p), dtype=int)
        for a in adj:
            adj_mask |= (np.array(a, dtype=int) != 0)
        comp_pvals = np.asarray(pvals_nbs)
    else:
        adj_mask = (np.array(adj, dtype=int) != 0)
        comp_pvals = np.atleast_1d(pvals_nbs)

    # --- group means in r-space and contrast ---
    mean_r_1 = np.tanh(Z_1.mean(axis=0))
    mean_r_2 = np.tanh(Z_2.mean(axis=0))
    cont_mat = mean_r_1 - mean_r_2

    # --- edgewise stats (Fisher-z inputs) ---
    iu = np.triu_indices(p, k=1)
    A_edges = Z_1[:, iu[0], iu[1]]  # (n1, E)
    B_edges = Z_2[:, iu[0], iu[1]]  # (n2, E)

    if paired:
        # match rows: Z_1 and Z_2 already aligned
        tvals, pvals = stats.ttest_rel(A_edges, B_edges, axis=0, nan_policy="omit")
    else:
        tvals, pvals = stats.ttest_ind(A_edges, B_edges, axis=0, equal_var=False, nan_policy="omit")

    # FDR (BH) over edges
    finite = np.isfinite(pvals)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    if finite.any():
        _, q_fdr, _, _ = multipletests(pvals[finite], alpha=0.05, method="fdr_bh")
        qvals[finite] = q_fdr

    # --- build full edge table ---
    df_all = pd.DataFrame({
        "roi_i": iu[0],
        "roi_j": iu[1],
        "roi_i_component": iu[0] + 1,  # 1-based for atlas dictionary
        "roi_j_component": iu[1] + 1,
        "mean_r_A": mean_r_1[iu],
        "mean_r_B": mean_r_2[iu],
        "delta_r_AminusB": cont_mat[iu],
        "t": tvals,
        "p": pvals,
        "q_fdr": qvals
    })

    # attach atlas labels if provided
    if atlas_dict_csv:
        atlas = pd.read_csv(atlas_dict_csv)
        atlas = atlas.rename(columns={
            atlas_component_col: "Component",
            atlas_name_col: "Difumo_names"
        })
        atlas["Component"] = atlas["Component"].astype(int)
        df_all = (df_all
                  .merge(atlas.add_suffix("_i"), how="left",
                         left_on="roi_i_component", right_on="Component_i")
                  .merge(atlas.add_suffix("_j"), how="left",
                         left_on="roi_j_component", right_on="Component_j"))
        # rename readable columns
        if "Difumo_names_i" in df_all.columns:
            df_all = df_all.rename(columns={"Difumo_names_i":"roi_i_name",
                                            "Difumo_names_j":"roi_j_name"})

    # --- NBS-selected subset (union mask) ---
    n_edges = int(adj_mask.sum() // 2)
    mask_flat = adj_mask[iu]
    df_sig = df_all.loc[mask_flat].copy()

    # --- save CSVs ---
    tag = f"{condition1}_{group1}_minus_{condition2}_{group2}".replace(" ", "")
    path_all = Path(out_dir, f"{tag}_edges_all.csv")
    path_sig = Path(out_dir, f"{tag}_edges_NBS_selected.csv")
    #df_all.to_csv(path_all, index=False)
    df_sig.to_csv(path_sig, index=False)

    # --- plots (same as before) ---
    adj_signed = cont_mat * adj_mask
    pos_mat = np.where(adj_signed > 0, adj_signed, 0.0)
    neg_mat = np.where(adj_signed < 0, adj_signed, 0.0)
    vmax = np.nanmax(np.abs(adj_signed));  vmax = 0.01 if (not np.isfinite(vmax) or vmax == 0) else vmax

    title_base = f"NBS: {condition1}-{group1} minus {condition2}-{group2}  (t>{thresh}, {k} perms)"

    # signed
    fig = plt.figure(figsize=(7.4, 6.4))
    plotting.plot_connectome(
        adjacency_matrix=adj_signed,
        node_coords=node_coords,
        edge_threshold=0.0,
        edge_vmin=-vmax, edge_vmax=vmax,
        edge_cmap=plt.cm.RdBu_r,
        node_size=18, colorbar=True,
        title=title_base,
        figure=fig
    )
    plt.savefig(Path(out_dir, f"{tag}_connectome_NBS_signed.png"), dpi=fig_dpi, bbox_inches="tight"); plt.close(fig)

    # pos
    if np.count_nonzero(pos_mat):
        fig = plt.figure(figsize=(7.4, 6.4))
        plotting.plot_connectome(
            adjacency_matrix=pos_mat,
            node_coords=node_coords,
            edge_threshold=0.0,
            edge_vmin=0.0, edge_vmax=vmax,
            node_size=18, colorbar=True,
            title=f"{title_base}  (positive)",
            figure=fig
        )
        plt.savefig(Path(out_dir, f"{tag}_connectome_NBS_pos.png"), dpi=fig_dpi, bbox_inches="tight"); plt.close(fig)

    # neg (magnitude)
    if np.count_nonzero(neg_mat):
        fig = plt.figure(figsize=(7.4, 6.4))
        plotting.plot_connectome(
            adjacency_matrix=np.abs(neg_mat),
            node_coords=node_coords,
            edge_threshold=0.0,
            edge_vmin=0.0, edge_vmax=vmax,
            node_size=18, colorbar=True,
            title=f"{title_base}  (negative magnitude)",
            figure=fig
        )
        plt.savefig(Path(out_dir, f"{tag}_connectome_NBS_neg.png"), dpi=fig_dpi, bbox_inches="tight"); plt.close(fig)

    # --- Cohen's d for NBS-selected edges ---
    if n_edges > 0:
        mask_flat_d = adj_mask[iu].astype(bool)
        A_sig = Z_1[:, iu[0][mask_flat_d], iu[1][mask_flat_d]]  # (n1, n_sig_edges)
        B_sig = Z_2[:, iu[0][mask_flat_d], iu[1][mask_flat_d]]  # (n2, n_sig_edges)
        
        if paired:
            # paired Cohen's d: mean(diff) / std(diff)
            diffs = A_sig - B_sig
            d_per_edge = diffs.mean(axis=0) / (diffs.std(axis=0, ddof=1) + 1e-9)
        else:
            var1 = A_sig.var(axis=0, ddof=1)
            var2 = B_sig.var(axis=0, ddof=1)
            pooled_sd = np.sqrt(((len(subs_1)-1)*var1 + (len(subs_2)-1)*var2) / 
                                (len(subs_1) + len(subs_2) - 2))
            d_per_edge = (A_sig.mean(axis=0) - B_sig.mean(axis=0)) / (pooled_sd + 1e-9)
        
        avg_cohen_d = float(np.mean(np.abs(d_per_edge)))
    else:
        avg_cohen_d = np.nan

    if verbose:
        print(f"[SAVE] {path_all.name}, {path_sig.name} in {out_dir}")
        print(f"[NBS] components={len(comp_pvals)} | pvals={np.array2string(comp_pvals, precision=3)}")
        print(f"[NBS] significant edges (union) = {n_edges}")

    return {
        "subs_1": subs_1,
        "subs_2": subs_2,
        "component_pvals": comp_pvals,
        "adj_mask_union": adj_mask.astype(int),
        "contrast_r": cont_mat,
        "edges_all_csv": str(path_all),
        "edges_nbs_csv": str(path_sig),
        "n_edges_selected": n_edges,
        "cohen_d": avg_cohen_d,
        "paired": paired,
        "labels": {"A": (condition1, group1), "B": (condition2, group2)},
    }


In [ ]:
interaction_tasks = [
    (D_ptsd, D_hc, "PTSD", "HC", "Full Script vs Hotspot")
]

results = Parallel(n_jobs=N_JOBS)(
    delayed(run_nbs_interaction)(
        Z_A=task[0], 
        Z_B=task[1], 
        label_A=task[2], 
        label_B=task[3], 
        condition_label=task[4],
        node_coords=node_coords,
        thresh=Threshold, 
        k=K,
        atlas_dict_csv=ATLAS_DIC
    ) for task in interaction_tasks
)

In [ ]:
results_w = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(run_nbs)(
        group1=group,
        condition1=CONDITIONS[0], 
        condition2=CONDITIONS[1],
        node_coords=node_coords,
        out_dir=OUT_DIR,
        thresh=5, 
        tail="both", 
        k=K, 
        paired=True,  
        fig_dpi=300, 
        verbose=False
    ) for group in GROUPS
)


In [ ]:
# --- Summary Table ---
print("\n" + "="*85)
print(f"{'Interaction':<30} | {'Sig?':<6} | {'P-val':<8} | {'Edges':<6} | {'Cohen d'}")
print("-" * 85)
for res in results:
    sig = "YES!*" if np.min(res['p_val']) < 0.05 else "no"
    print(f"{res['comparison']:<30} | {sig:<6} | {res['p_val']:.4f} | {res['n_edges']:<6} | {res['avg_d']:.3f}")
print("\n" + "="*85)

In [ ]:
# --- Summary Table ---
print("\n" + "="*100)
print(f"{'GROUP A':<15} | {'GROUP B':<15} | {'SIG?':<6} | {'P-VAL':<6} | {'EDGES':<6} | {'Cohen d':<8}")
print("-" * 100)
for res in results_w:
    label_a = f"{res['labels']['A'][0]}_{res['labels']['A'][1]}"
    label_b = f"{res['labels']['B'][0]}_{res['labels']['B'][1]}"
    pvals = np.mean(res['component_pvals'])
    d_str = f"{res['cohen_d']:.3f}"
    sig = "YES!*" if pvals < 0.05 else "no"
    print(f"{label_a:<15} | {label_b:<15} | {sig:<6} | {pvals:.4f} | {res['n_edges_selected']:<6} | {d_str:<8}")
print("="*100 + "\n")

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w -p xarray